# Robustness and Sensitivity Walkthrough

This notebook reproduces the robustness checks used to assess the stability of the stage-based findings. It loads the generated CSV outputs rather than re-fitting models interactively, so it can be executed quickly by artifact evaluators.

In [1]:
from pathlib import Path
import pandas as pd

# Resolve repository root even when Jupyter starts in a different working directory.
ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "RQ2_Prompt_Effectiveness_Modeling").exists() and (candidate / "Dataset_Construction").exists():
        ROOT = candidate
        break

results = pd.read_csv(ROOT / "RQ2_Prompt_Effectiveness_Modeling/results/diagnostics/full_robustness_sensitivity_results.csv")
exclusions = pd.read_csv(ROOT / "RQ2_Prompt_Effectiveness_Modeling/results/diagnostics/robustness_exclusion_details.csv")
results.head()

,Family,Gate,Scenario,Specification,N,Term,Metric,Estimate,CI_Low,CI_High,p_value,Significant_0_05,Direction,Formatted
0,Repository-level dependence,Gate 0,Repository-clustered SE,C/S/V + PR size,218,Context,OR,2.143088,1.090196,4.212846,0.027078,True,positive,"2.14* [1.09, 4.21]"
1,Repository-level dependence,Gate 0,Repository-clustered SE,C/S/V + PR size,218,Specificity,OR,65.838382,9.230535,469.603605,0.000030,True,positive,"65.84*** [9.23, 469.60]"
2,Repository-level dependence,Gate 0,Repository-clustered SE,C/S/V + PR size,218,Verification,OR,0.903855,0.418553,1.951852,0.796908,False,negative,"0.90 [0.42, 1.95]"
3,Repository-level dependence,Gate 0,Repository-clustered SE,C/S/V + PR size,218,Log_PR_Size,OR,1.115435,0.906835,1.372018,0.301054,False,positive,"1.12 [0.91, 1.37]"
4,Repository-level dependence,Gate 1,Repository-clustered SE,C/S/V + PR size,141,Context,OR,2.218866,1.002313,4.912009,0.049338,True,positive,"2.22* [1.00, 4.91]"


## Exclusion details

Each sensitivity check is applied within the gate-specific modeling sample. This avoids using a repository or language that is dominant in the full dataset but not dominant for a specific gate.

In [2]:
exclusions

,Family,Gate,Scenario,N_before,N_after,Removed
0,Dominant repository sensitivity,Gate 0,Exclude largest repository,218,208,VOICEVOX/voicevox
1,Dominant repository sensitivity,Gate 0,Exclude top 5 repositories,218,188,VOICEVOX/voicevox; UNLV-CS472-672/2024-S-GROUP...
2,Dominant repository sensitivity,Gate 1,Exclude largest repository,141,135,UNLV-CS472-672/2024-S-GROUP3-Barbell
3,Dominant repository sensitivity,Gate 1,Exclude top 5 repositories,141,121,UNLV-CS472-672/2024-S-GROUP3-Barbell; open-lea...
4,Dominant repository sensitivity,Gate 2,Exclude largest repository,89,83,UNLV-CS472-672/2024-S-GROUP3-Barbell
5,Dominant repository sensitivity,Gate 2,Exclude top 5 repositories,89,71,UNLV-CS472-672/2024-S-GROUP3-Barbell; UNLV-CS4...
6,Language sensitivity,Gate 0,Exclude dominant language,218,158,TypeScript
7,Language sensitivity,Gate 0,Exclude top 2 languages,218,127,TypeScript; Python
8,Language sensitivity,Gate 1,Exclude dominant language,141,102,TypeScript
9,Language sensitivity,Gate 1,Exclude top 2 languages,141,79,TypeScript; Python


## Main individual-dimension models

The table below reports the Context, Specificity, and Verification results for each gate and sensitivity scenario. Gate 0 and Gate 1 report odds ratios; Gate 2 reports approximate average marginal effects from the fractional-logit sensitivity model.

In [3]:
results[results["Specification"] == "CSV dimensions"][["Gate", "Scenario", "N", "Term", "Metric", "Formatted"]]

,Gate,Scenario,N,Term,Metric,Formatted


## Aggregate PQS sensitivity

These models replace the individual prompt dimensions with aggregate PQS. They are useful as a robustness comparison, but the paper emphasizes individual dimensions because they preserve interpretability.

In [5]:
results[results["Specification"] == "Aggregate PQS"][["Gate", "Scenario", "N", "Term", "Metric", "Formatted"]]

,Gate,Scenario,N,Term,Metric,Formatted


## Interpretation

The results support qualitative stability of the paper findings. The only notable attenuation is Gate 2 under dominant-language exclusion, where removing TypeScript reduces the PA-only sample to 57 cases and widens the confidence interval for Context.